# 04 — Video: LMVD Audio + Visual LSTM


In [ ]:
import sys
from pathlib import Path
ROOT = Path('..').resolve()
sys.path.insert(0, str(ROOT))

import subprocess
import matplotlib.pyplot as plt
import numpy as np
DATA_DIR = ROOT / "lmvd-analysis" / "data"
TCN_DIR = DATA_DIR / "tcnfeature"
AUDIO_DIR = DATA_DIR / "audiofeature"
print("Video features:", TCN_DIR.exists(), "| Audio:", AUDIO_DIR.exists())


In [ ]:
if not (TCN_DIR.exists() and any(TCN_DIR.glob("*.npy"))):
    print("Downloading/extracting LMVD features (~16 GB, resumable)...")
    subprocess.run([sys.executable, str(ROOT / "lmvd-analysis" / "download_lmvd_features.py")], cwd=str(ROOT))
else:
    print(f"Ready: {len(list(TCN_DIR.glob('*.npy')))} video, {len(list(AUDIO_DIR.glob('*.npy')))} audio")


In [ ]:
from src.data.loaders import load_lmvd_labels
labels = load_lmvd_labels(DATA_DIR)
print(labels["class"].value_counts())
print(labels.groupby("split")["label"].mean().round(3))


In [ ]:
zip_path = DATA_DIR / "LMVD_Feature.zip"
has_zip = zip_path.exists() and zip_path.stat().st_size > 15_000_000_000
has_extracted = TCN_DIR.exists() and any(TCN_DIR.glob("*.npy"))

if not has_extracted and has_zip:
    print("Using LMVD features directly from zip (no full extraction — saves disk space)")
    from src.data.loaders import load_lmvd_sequence
    v, a = load_lmvd_sequence(501, DATA_DIR)
    print(f"Sample 501 from zip: video {v.shape}, audio nonzero={np.any(a)}")
elif not has_extracted:
    print("LMVD zip not complete. Run: python lmvd-analysis/download_lmvd_features.py")
else:
    print(f"Extracted features: {len(list(TCN_DIR.glob('*.npy')))} video files")


In [ ]:
if has_zip or has_extracted:
    import torch
    from torch.utils.data import DataLoader
    from src.models.video_lstm import LMVDSequenceDataset, VideoAudioLSTM, train_video_lstm
    from src.utils import load_config, set_seed
    from scripts.extract_lmvd_modalities import extract_and_cache_samples
    set_seed(42)
    cfg = load_config(ROOT / "config.yaml")["video_models"]

    # Stratified subset (label.head(N) alone would grab a single-class block —
    # the labels frame is grouped by class) + make sure those exact sample_ids
    # are actually cached to disk before building datasets, since
    # LMVDSequenceDataset(cache_only=True) silently returns all-zero features
    # for anything not already extracted from the zip.
    def _stratified(df, n):
        per_class = max(1, n // 2)
        return df.groupby("label", group_keys=False).apply(lambda g: g.head(per_class)).head(n)

    train_df = _stratified(labels[labels["split"] == "train"], 200)
    val_df = _stratified(labels[labels["split"] == "val"], 50)

    print("Extracting/caching train+val samples from the LMVD zip (one-time, streamed - no full unzip)...")
    train_ids = extract_and_cache_samples(train_df["sample_id"].tolist(), DATA_DIR)
    val_ids = extract_and_cache_samples(val_df["sample_id"].tolist(), DATA_DIR)
    train_df = train_df[train_df["sample_id"].isin(train_ids)]
    val_df = val_df[val_df["sample_id"].isin(val_ids)]
    print(f"Cached {len(train_df)} train / {len(val_df)} val samples "
          f"(train label balance: {train_df['label'].value_counts().to_dict()})")

    train_loader = DataLoader(LMVDSequenceDataset(train_df["sample_id"].tolist(), train_df["label"].tolist(), DATA_DIR),
                              batch_size=8, shuffle=True)
    val_loader = DataLoader(LMVDSequenceDataset(val_df["sample_id"].tolist(), val_df["label"].tolist(), DATA_DIR),
                            batch_size=8)
    model = VideoAudioLSTM(hidden_dim=64, num_layers=1, dropout=0.3)
    history = train_video_lstm(model, train_loader, val_loader, epochs=5, lr=1e-3)
    print(f"Best val accuracy: {max(history['val_acc']):.4f}")
    plt.figure(figsize=(8, 3))
    plt.plot(history["val_acc"]); plt.title("LMVD LSTM Val Accuracy"); plt.tight_layout()
    plt.savefig(ROOT / "outputs" / "lmvd_lstm_training.png", dpi=150); plt.close()
    subprocess.run([sys.executable, str(ROOT / "scripts" / "extract_lmvd_modalities.py"), "--max-samples", "20"], cwd=str(ROOT))
else:
    print("LMVD features not ready yet — label stats printed above.")

print("04_video_lmvd.py complete.")
